# Data Preprocessing

Flatten collected YouTube comments and replies into rows for sentiment analysis and network analysis.


In [31]:
import json
from pathlib import Path
import pandas as pd
import random 
from langdetect import detect, LangDetectException
from collections import Counter
import nltk
import string
import re
import html
import unicodedata
import emoji
from nltk.corpus import stopwords
from itertools import combinations


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
VIDEO_DATA_PATH = DATA_DIR / "video_data.json"
PROCESSED_VIDEO_DATA_PATH = DATA_DIR / "video_data_processed.json"
MET_GALA_ENTITIES_PATH = DATA_DIR / "met_gala_entities.json"

RANDOM_SEED = 42

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)


True

In [4]:
with open(VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    video_data = json.load(f)["videos"]

print("Video data loaded")
print("Total videos:", len(video_data))
print("Total collected comment rows:", sum(len(video.get("comments", [])) for video in video_data))


Video data loaded
Total videos: 110
Total collected comment rows: 63250


In [5]:
comments_flattened = []
for video in video_data:
    video_context = {
        "video_id": video.get("videoId"),
        "video_title": video.get("title"),
        "channel_id": video.get("channelId"),
        "channel_title": video.get("channelTitle"),
        "video_published_at": video.get("publishedAt"),
        "video_view_count": video.get("viewCount", 0),
        "video_like_count": video.get("likeCount", 0),
        "video_available_comment_count": video.get("commentCount", 0),
    }

    for comment in video.get("comments", []):
        comments_flattened.append({
            **video_context,
            "comment_id": comment.get("commentId"),
            "comment_text": comment.get("text", ""),
            "comment_author_id": comment.get("authorId"),
            "comment_author": comment.get("author"),
            "comment_published_at": comment.get("publishedAt"),
            "comment_updated_at": comment.get("updatedAt"),
            "comment_like_count": comment.get("likeCount", 0),
            "is_reply": comment.get("isReply", False),
            "parent_comment_id": comment.get("parentCommentId"),
            "reply_to_author_id": comment.get("replyToAuthorId"),
            "top_level_reply_count": comment.get("totalReplyCount", 0),
            "text_length": len(comment.get("text", "") or ""),
        })

total_comments = len(comments_flattened)
total_replies = sum(1 for c in comments_flattened if c.get("is_reply") == True)
total_parent_comments = total_comments - total_replies

print(f"Flattened comment rows: {total_comments}\n")

print(f"Total comments: {total_comments}")
print(f"Total parent comments: {total_parent_comments}")
print(f"Total replies: {total_replies}")


Flattened comment rows: 63250

Total comments: 63250
Total parent comments: 48960
Total replies: 14290


In [6]:

TOTAL_RANDOM_SAMPLES = 25

print("\nRANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES")
print("=" * 80)
random.seed(RANDOM_SEED)
random_sample_indices = random.sample(range(len(comments_flattened)), min(TOTAL_RANDOM_SAMPLES, len(comments_flattened)))
for i, idx in enumerate(random_sample_indices):
    text = comments_flattened[idx]['comment_text'].strip().replace('\n', ' ').replace('\r', '')
    text = ' '.join(text.split())
    print(f"[{i+1}/{TOTAL_RANDOM_SAMPLES}] {text[:150]:<10}")



RANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES
[1/25] Can’t be bothered to pronounce the Asian names properly I guess.
[2/25] 14:30 in my opinion, it needed a big sumptuous cloak, maybe with a hood and gloves, to be more evocative of how luscious and involved klimt’s pieces a
[3/25] what is cara doing :/
[4/25] We need you at the Grammy asap😂😂
[5/25] @LadyAxe13 🤣
[6/25] Megyn is sooo jealous she wasn't invited
[7/25] My only look at the met gala, thanks Garrron! Getting Halloween in springtime vibes 🤡👻👽👀
[8/25] I agree with so many of your critiques. The fact Anna didn't even bother with the theme and wore a variation of a previous dress let's me know how fri
[9/25] I be sick of them shades. You can't wear shades with everything. I was just thinking this when I first seen this video. She look like one of the actor
[10/25] NINGNING  
[11/25] Vogue deletes comments and leaves hates comments on him.
[12/25] Exactly!  
[13/25] The idea is worth sharing This deserves recognition.
[14/25] Bla

In [7]:
def detect_language(text):
    """Detect language, returning ISO code."""
    try:
        if not text or not len(text.strip()):
            return 'en'
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Collect comment rows from the rows list
language_results = [(comment, detect_language(comment.get('comment_text', ''))) for comment in comments_flattened]
language_counter = Counter(lang for _, lang in language_results)

In [8]:
TOP_LANGUAGE_COUNT = 10
TOTAL_NON_ENGLISH_EXAMPLES = 20

total_comments = len(language_results)

print(f"\nTOP {TOP_LANGUAGE_COUNT} LANGUAGE DETECTIONS")
print("=" * 80)
for i, (lang, count) in enumerate(language_counter.most_common(TOP_LANGUAGE_COUNT), start=1):
    pct = 100 * count / total_comments
    print(f"[{i}] {lang} {count:,} ({pct:.2f}%)")
    if i == 10:
        break


TOP 10 LANGUAGE DETECTIONS
[1] en 46,920 (74.18%)
[2] unknown 1,679 (2.65%)
[3] so 1,563 (2.47%)
[4] pt 1,135 (1.79%)
[5] de 1,071 (1.69%)
[6] tl 1,057 (1.67%)
[7] af 957 (1.51%)
[8] fr 847 (1.34%)
[9] et 803 (1.27%)
[10] id 771 (1.22%)


In [9]:
COMMENT_TRUNCATION_LENGTH = 200
ENGLISH_FILTER = "en"

# Extract full comment data for English comments
english_comments = [comment for comment, lang in language_results if lang == ENGLISH_FILTER]
for comment in english_comments:
    comment["language"] = ENGLISH_FILTER

# Overwrite comments flattened with English filtered list
comments_flattened = english_comments

# Extract truncated comment data for non-English comments for test display
non_english_comments = [comment['comment_text'][:COMMENT_TRUNCATION_LENGTH] for comment, lang in language_results if lang != 'en']

random.seed(RANDOM_SEED)
random_non_english = random.sample(non_english_comments, min(TOTAL_NON_ENGLISH_EXAMPLES, len(non_english_comments)))
random_english = random.sample(comments_flattened, min(TOTAL_NON_ENGLISH_EXAMPLES, len(comments_flattened)))

In [10]:

print("\nRANDOM NON-ENGLISH COMMENTS REMOVED:")
print("=" * 80)
for idx, text in enumerate(random_non_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {text}")



RANDOM NON-ENGLISH COMMENTS REMOVED:
[1/20] 💯
[2/20] 🙋‍♀️❤️
[3/20] Jisoo jennie lisaa rose
[4/20] Elegante, hermoso y talentoso😊😊😊❤❤❤
[5/20] Лlol😅
[6/20] PedoFest 2026
[7/20] Habsburg Jaw
[8/20] Jisoo 💖
[9/20] Devika didi army under 15 minutes 😂
[10/20] Omg rosé gorgeussss❤❤❤
[11/20] linda
[12/20] ❤❤❤❤❤❤❤❤❤❤❤❤
[13/20] Alwaaays love her interviews 🥰
[14/20] Fr
[15/20] 32:08 Jisoo interview
[16/20] Blake still here
[17/20] 🤢🤢🤮🤮🤮🤮🤮🤮🤮🤮
[18/20] وابزانم وادەزانن ئینگلیزی دڵم 😂😂😂
[19/20] 24:00 ew
[20/20] Lmfao


In [11]:
print("\nRANDOM ENGLISH COMMENTS REMAINING:")
print("=" * 80)
for idx, comment in enumerate(random_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {comment['comment_text'][:COMMENT_TRUNCATION_LENGTH]}")



RANDOM ENGLISH COMMENTS REMAINING:
[1/20] Riri has graced us, I love her vibe with Asap! To me that moment is stolen and snatched by Riri and Asap…Period!!!
[2/20] No Madonna? Do u hate her or something?
[3/20] i can listen to these two talk for hours
[4/20] ​@kvibez_random yeah I get that ❤
[5/20] I’m cryinnnnn it comes to me lateeee😭😭😭😭
[6/20] Was I the only one who thought Bad Bunny was referencing the Portrait of Dorian Gray?
[7/20] Anna Wintour is British and they have been actively trying to destroy our culture.
[8/20] Someone please teach Emma how to stand.
[9/20] Tyla the chosen one 👌 This woman never fails.
[10/20] Beyonce babies don't be having it .....Blu informed them .."I'll take em off when I go up there "..and my Rumi Love 💝...you already know..Rumi pulls away from the staff....and Mama Tina ..as if to say
[11/20] The purple version without all the stringy hair would have been much better.
[12/20] I can’t wait to see that list! 😂
[13/20] Racheal's jaw? Thinking maybe a 

In [12]:
english_count = len(english_comments)
removed_count = total_comments - english_count
english_pct = 100 * english_count / total_comments
removed_pct = 100 * removed_count / total_comments

print(f"\nEnglish kept: {english_count} ({english_pct:.1f}%)")
print(f"Non-English removed: {removed_count} ({removed_pct:.1f}%)")


English kept: 46920 (74.2%)
Non-English removed: 16330 (25.8%)


In [13]:
# Regex patterns shared by the cleaning helpers
URL_PATTERN = re.compile(r'https?://\S+')
TIMESTAMP_PATTERN = re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b')
MENTION_PATTERN = re.compile(r'@[\w.-]+[\w]')
DIGIT_PATTERN = re.compile(r'\d+')
PUNCT_PATTERN = re.compile(r'[^\w\s]')
ENTITY_SEPARATOR_PATTERN = re.compile(r'[^a-z\s]+')


In [14]:
# Light cleaning for entity matching and network basic network assessment
def remove_html_entities(text):
    return html.unescape(text or "")

def remove_urls(text):
    return URL_PATTERN.sub("", text)

def remove_timestamps(text):
    return TIMESTAMP_PATTERN.sub("", text)

def remove_mentions(text):
    return MENTION_PATTERN.sub("", text)

def remove_accents(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()

def strip_entity_separators(text):
    return ENTITY_SEPARATOR_PATTERN.sub(" ", text)

def normalise_whitespace(text):
    return " ".join(text.split())

def clean_for_entity_matching(text):
    text = remove_html_entities(text)
    text = remove_urls(text)
    text = remove_timestamps(text)
    text = remove_mentions(text)
    text = remove_accents(text.lower())
    text = strip_entity_separators(text)
    return normalise_whitespace(text)

def sentiment_analysis_clean(text):
    """Minimal cleaning appropraite for VADER and BERT sentiment analysis"""
    text = remove_html_entities(text)
    text = remove_urls(text)
    text = remove_timestamps(text)
    text = remove_mentions(text)
    return normalise_whitespace(text)

# Store minimally processed text
# Depending on sentiment approach, this might be enough processing
for comment in comments_flattened:
    comment["comment_text_entity"] = clean_for_entity_matching(comment.get("comment_text", ""))

print("Lightly cleaned text ready for entity matching")


Lightly cleaned text ready for entity matching


In [15]:
unique_videos = set(comment['video_id'] for comment in comments_flattened)
unique_channels = set(comment['channel_id'] for comment in comments_flattened)
unique_authors = set(comment['comment_author'] for comment in comments_flattened)

SHORT_COMMENT_LENGTH = 8
short_comments = [comment for comment in comments_flattened if len(comment['comment_text']) < SHORT_COMMENT_LENGTH]
comment_lengths = [len(comment['comment_text']) for comment in comments_flattened]

comments_with_urls = [c for c in comments_flattened if URL_PATTERN.search(c['comment_text'])]
comments_with_timestamps = [c for c in comments_flattened if TIMESTAMP_PATTERN.search(c['comment_text'])]
comments_with_mentions = [c for c in comments_flattened if MENTION_PATTERN.search(c['comment_text'])]
comments_with_digits = [c for c in comments_flattened if DIGIT_PATTERN.search(c['comment_text'])]
comments_with_punctuation = [c for c in comments_flattened if PUNCT_PATTERN.search(c['comment_text'])]

comments_with_urls_pct = 100 * len(comments_with_urls) / len(comments_flattened)
comments_with_timestamps_pct = 100 * len(comments_with_timestamps) / len(comments_flattened)
comments_with_mentions_pct = 100 * len(comments_with_mentions) / len(comments_flattened)
comments_with_digits_pct = 100 * len(comments_with_digits) / len(comments_flattened)
comments_with_punctuation_pct = 100 * len(comments_with_punctuation) / len(comments_flattened)

# For author, video, and channel distributions
video_counter = Counter(comment['video_id'] for comment in comments_flattened)
channel_counter = Counter(comment['channel_id'] for comment in comments_flattened)
author_counter = Counter(comment['comment_author'] for comment in comments_flattened)

# Print all details at bottom
print("BASIC DATA EXPLORATION")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Unique videos: {len(unique_videos)}")
print(f"Unique channels: {len(unique_channels)}")
print(f"Unique authors: {len(unique_authors)}\n")

print(f"Short comments (<{SHORT_COMMENT_LENGTH} chars): {len(short_comments)}")
print(f"Comments w/ URLs: {len(comments_with_urls)} ({comments_with_urls_pct:.2f}%)")
print(f"Comments w/ timestamps: {len(comments_with_timestamps)} ({comments_with_timestamps_pct:.2f}%)")
print(f"Comments w/ mentions: {len(comments_with_mentions)} ({comments_with_mentions_pct:.2f}%)")
print(f"Comments w/ digits: {len(comments_with_digits)} ({comments_with_digits_pct:.2f}%)")
print(f"Comments w/ punctuation: {len(comments_with_punctuation)}\n")

print(f"Max comment length: {max(comment_lengths) if comment_lengths else 0}")
print(f"Average comment length: {sum(comment_lengths)/len(comment_lengths):.2f}" if comment_lengths else "Avg. comment length: 0")

BASIC DATA EXPLORATION
Total comments: 46920
Unique videos: 109
Unique channels: 73
Unique authors: 35131

Short comments (<8 chars): 234
Comments w/ URLs: 19 (0.04%)
Comments w/ timestamps: 1463 (3.12%)
Comments w/ mentions: 3761 (8.02%)
Comments w/ digits: 6912 (14.73%)
Comments w/ punctuation: 40156

Max comment length: 9817
Average comment length: 97.77


In [16]:
with open(MET_GALA_ENTITIES_PATH, "r", encoding="utf-8") as f:
    met_gala_entities = json.load(f)

total_entities = len(met_gala_entities['entities'])
celebs = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'celebrity']
brands = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'designer_brand']

print(f"Total entities: {total_entities}")
print(f"Total celebrities: {len(celebs)}")
print(f"Total brands: {len(brands)}")

Total entities: 483
Total celebrities: 361
Total brands: 122


In [17]:
# Prepare exact entity aliases after light cleaning is available
def build_entity_aliases(entities):
    entity_aliases = []
    for entity in entities:
        aliases = set()
        for alias in entity.get("aliases", []) + [entity["name"]]:
            clean_alias = clean_for_entity_matching(alias)
            if clean_alias:
                aliases.add(clean_alias)
        entity_aliases.append({"name": entity["name"], "aliases": sorted(aliases)})
    return entity_aliases

def find_entities(text, entity_aliases):
    clean_text = clean_for_entity_matching(text)
    padded_text = f" {clean_text} "
    found = []
    for entity in entity_aliases:
        for alias in entity["aliases"]:
            if f" {alias} " in padded_text:
                found.append(entity["name"])
                break
    return found

# Create exact alias matching for celeb and brand names
celeb_aliases = build_entity_aliases(celebs)
brand_aliases = build_entity_aliases(brands)


In [18]:
# Match celebrity and brand aliases in each comment
matched_comments = []
celeb_counter = Counter()
brand_counter = Counter()

for comment in comments_flattened:
    text_for_matching = comment.get("comment_text_entity", "")

    # Find entities in comment
    found_celebs = find_entities(text_for_matching, celeb_aliases)
    found_brands = find_entities(text_for_matching, brand_aliases)

    # Scoring on each counter
    for celeb in found_celebs:
        celeb_counter[celeb] += 1
    for brand in found_brands:
        brand_counter[brand] += 1

    comment["celebs"] = found_celebs
    comment["brands"] = found_brands

    # Append to matched comments
    matched_comments.append({
        **comment,
        "comment_id": comment.get("comment_id"),
        "video_id": comment.get("video_id"),
        "video_title": comment.get("video_title"),
        "comment_text": comment.get("comment_text", ""),
        "comment_text_entity": text_for_matching,
        "celebs": found_celebs,
        "brands": found_brands,
    })


In [19]:
comments_with_celebs = [row for row in matched_comments if row["celebs"]]
comments_with_brands = [row for row in matched_comments if row["brands"]]
# Which comments have both brand and celeb mentions 
# Expect this to be smaller
comments_with_both = [row for row in matched_comments if row["celebs"] and row["brands"]]

comments_with_celebs_pct = 100 * len(comments_with_celebs) / len(comments_flattened)
comments_with_brands_pct = 100 * len(comments_with_brands) / len(comments_flattened)
comments_with_both_pct = 100 * len(comments_with_both) / len(comments_flattened)

print("ENTITY MATCH COVERAGE")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Comments with celebrity mentions: {len(comments_with_celebs)} ({comments_with_celebs_pct:.2f}%)")
print(f"Comments with brand mentions: {len(comments_with_brands)} ({comments_with_brands_pct:.2f}%)")
print(f"Comments with both celebrity and brand mentions: {len(comments_with_both)} ({comments_with_both_pct:.2f}%)")


ENTITY MATCH COVERAGE
Total comments: 46920
Comments with celebrity mentions: 10326 (22.01%)
Comments with brand mentions: 1027 (2.19%)
Comments with both celebrity and brand mentions: 357 (0.76%)


In [20]:
# Entities that are being found often enough to work with
matched_celeb_count = len(celeb_counter)
matched_brand_count = len(brand_counter)
matched_celeb_count_pct = 100 * matched_celeb_count / len(celebs)
matched_brand_count_pct = 100 * matched_brand_count / len(brands)

print("ENTITY FREQUENCY CHECK")
print("=" * 80)
print(f"Matched celebrities: {matched_celeb_count}/{len(celebs)} ({matched_celeb_count_pct:.2f}%)")
print(f"Matched brands: {matched_brand_count}/{len(brands)} ({matched_brand_count_pct:.2f}%)")


ENTITY FREQUENCY CHECK
Matched celebrities: 216/361 (59.83%)
Matched brands: 68/122 (55.74%)


In [21]:

print("\nTOP 20 CELEBRITIES")
print("=" * 80)
for celeb, count in celeb_counter.most_common(20):
    print(f"{celeb}: {count}")



TOP 20 CELEBRITIES
Beyonce: 1336
Jisoo: 1299
LISA: 1054
Rose: 943
Rihanna: 749
Madonna: 435
JENNIE: 413
Emma Chamberlain: 372
Heidi Klum: 338
Cardi B: 332
Anne Hathaway: 307
Kylie Jenner: 242
Katy Perry: 229
Bad Bunny: 222
Blake Lively: 214
Sabrina Carpenter: 202
Karan Johar: 200
Tyla: 183
Sam Smith: 182
Ningning: 174


In [22]:

print("\nTOP 20 BRANDS")
print("=" * 80)
for brand, count in brand_counter.most_common(20):
    print(f"{brand}: {count}")


TOP 20 BRANDS
Saint Laurent: 259
Robert Wun: 161
Dior: 87
Mugler: 77
Chanel: 70
Hugo Boss: 52
Balenciaga: 43
Schiaparelli: 40
Prada: 34
Zara: 24
Skims: 21
Allen Jones: 19
Zac Posen: 17
Gap Studio: 16
Valentino: 15
Chloe: 12
Tom Ford: 10
Christian Siriano: 9
Michael Kors: 9
Vivienne Westwood: 9


In [23]:
TUPLE_SIZE = 2

# Count occurrences of pairs of celebrities mentioned together
celeb_pair_counter = Counter()
for comment in comments_flattened:
    celebs = sorted(set(comment.get("celebs", [])))
    if len(celebs) > 1:
        for pair in combinations(sorted(celebs), TUPLE_SIZE):
            celeb_pair_counter[pair] += 1

In [24]:

print("\nTOP 20 CELEBRITY PAIRS")
print("=" * 80)
for pair, count in celeb_pair_counter.most_common(20):
    print(f"{pair}: {count}")



TOP 20 CELEBRITY PAIRS
('Jisoo', 'Rose'): 324
('Jisoo', 'LISA'): 292
('JENNIE', 'LISA'): 196
('JENNIE', 'Rose'): 178
('JENNIE', 'Jisoo'): 174
('LISA', 'Rose'): 169
('Beyonce', 'Rihanna'): 103
('Karina', 'Ningning'): 63
('Beyonce', 'LISA'): 46
('Jisoo', 'Ningning'): 46
('Beyonce', 'Madonna'): 42
('Jisoo', 'Karina'): 41
('Beyonce', 'Emma Chamberlain'): 36
('Beyonce', 'Jay-Z'): 34
('Beyonce', 'Sabrina Carpenter'): 33
('Anne Hathaway', 'Beyonce'): 33
('LISA', 'Ningning'): 33
('Beyonce', 'Blue Ivy'): 32
('Isha Ambani', 'Karan Johar'): 31
('Beyonce', 'Tyla'): 30


In [25]:
# Count tuple pairs of (brand, celeb), retaining supporting comments
brand_celeb_counter = Counter()
brand_celeb_comment_ids = {}
for row in comments_with_both:
    for brand in row["brands"]:
        for celeb in row["celebs"]:
            pair = (brand, celeb)
            brand_celeb_counter[pair] += 1
            brand_celeb_comment_ids.setdefault(pair, []).append(row.get("comment_id"))

# Create edge rows
edge_rows = []
for (brand, celeb), count in brand_celeb_counter.items():
    edge_rows.append({
        "source": brand,
        "target": celeb,
        "source_type": "brand",
        "target_type": "celebrity",
        "weight": count,
        "comment_ids": brand_celeb_comment_ids.get((brand, celeb), []),
    })

# Higher weight better, stronger indicator
edge_rows_sorted = sorted(edge_rows, key=lambda x: x["weight"], reverse=True)

In [26]:
TOTAL_BRAND_CELEBRITY_EXAMPLES = 40

print("BRAND-CELEBRITY EDGE CHECK")
print("=" * 80)
print(f"Unique brand-celebrity edges: {len(edge_rows_sorted)}")
print(f"Total brand-celebrity co-mentions: {sum(brand_celeb_counter.values())}")

BRAND-CELEBRITY EDGE CHECK
Unique brand-celebrity edges: 427
Total brand-celebrity co-mentions: 843


In [27]:

print("\nTOP 20 BRAND-CELEBRITY EDGES")
BRAND_CELEBRITY_PAD = 40
WEIGHT_PAD = 10
print(f"{'(BRAND, CELEBRITY)':<{BRAND_CELEBRITY_PAD}} {'WEIGHT':<{WEIGHT_PAD}}")
print("=" * (BRAND_CELEBRITY_PAD + 1 + WEIGHT_PAD))
for edge in edge_rows_sorted[:TOTAL_BRAND_CELEBRITY_EXAMPLES]:
    source_target = (edge['source'], edge['target'])
    print(f"{str(source_target):<{BRAND_CELEBRITY_PAD}} {edge['weight']:<{WEIGHT_PAD}}")



TOP 20 BRAND-CELEBRITY EDGES
(BRAND, CELEBRITY)                       WEIGHT    
('Saint Laurent', 'Rose')                61        
('Dior', 'Jisoo')                        30        
('Robert Wun', 'LISA')                   20        
('Mugler', 'Emma Chamberlain')           18        
('Saint Laurent', 'Jisoo')               14        
('Dior', 'LISA')                         11        
('Saint Laurent', 'Connor Storrie')      11        
('Saint Laurent', 'LISA')                11        
('Chanel', 'JENNIE')                     10        
('Saint Laurent', 'JENNIE')              10        
('Robert Wun', 'Naomi Osaka')            8         
('Saint Laurent', 'Madonna')             7         
('Dior', 'JENNIE')                       7         
('Dior', 'Rose')                         7         
('Balenciaga', 'Beyonce')                7         
('Robert Wun', 'Audrey Nuna')            6         
('Vivienne Westwood', 'Jeremy Pope')     6         
('Schiaparelli', 'Lauren Sanchez B

In [28]:
# Basic inspection of nodes and edges
nodes = []
for celeb, count in celeb_counter.items():
    nodes.append({
        "node": celeb, 
        "type": "celebrity", 
        "mention_count": count
    })
for brand, count in brand_counter.items():
    nodes.append({
        "node": brand, 
        "type": "brand", 
        "mention_count": count
    })
nodes_sorted = sorted(nodes, key=lambda x: x["mention_count"], reverse=True)

print("BASIC NETWORK ASSESSMENT")
print("=" * 80)
print(f"Nodes available: {len(nodes_sorted)}")
print(f"Edges available: {len(edge_rows_sorted) if 'edge_rows_sorted' in locals() else 0}")
print(f"Comments supporting edges: {len(comments_with_both)}")
print(f"Edges with weight >= 2: {sum(1 for edge in edge_rows_sorted if edge['weight'] >= 2) if 'edge_rows_sorted' in locals() else 0}")


BASIC NETWORK ASSESSMENT
Nodes available: 284
Edges available: 427
Comments supporting edges: 357
Edges with weight >= 2: 124


In [32]:
TWEET_TOKENISER = nltk.tokenize.TweetTokenizer(
    reduce_len=True,
    strip_handles=True,
    preserve_case=False 
)

PUNCTUATION = list(string.punctuation)
TWEET_STEMMER = nltk.stem.PorterStemmer()
STOP_WORDS = set(stopwords.words('english')) | set(PUNCTUATION)


In [33]:
# Heavier cleaning for topic preprocessing
def remove_digits(text):
    return DIGIT_PATTERN.sub(" ", text)

def remove_unicode(text):
    return text.encode("ascii", "ignore").decode()

def strip_punctuation(text):
    return PUNCT_PATTERN.sub(" ", text)

def tokenize(text):
    return TWEET_TOKENISER.tokenize(text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS]

def stem_tokens(tokens):
    return [TWEET_STEMMER.stem(t) for t in tokens]

def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

def clean_for_topic_text(text):
    text = clean_for_entity_matching(text)
    text = remove_unicode(text)
    text = remove_digits(text)
    text = strip_punctuation(text)
    return normalise_whitespace(text)

def clean_for_topic_tokens(text):
    text = clean_for_topic_text(text)
    tokens = tokenize(text)
    return remove_stopwords(tokens)

def clean_for_topic_stemmed_tokens(text):
    return stem_tokens(clean_for_topic_tokens(text))

def purify_text(text, show_changes=False):
    """Return heavily cleaned, tokenized text with stopwords removed."""
    if not show_changes:
        return clean_for_topic_tokens(text)

    history = {}
    text = remove_html_entities(text)
    history["remove_html_entities"] = text
    text = remove_urls(text)
    history["remove_urls"] = text
    text = remove_timestamps(text)
    history["remove_timestamps"] = text
    text = remove_mentions(text)
    history["remove_mentions"] = text
    text = text.lower().strip()
    history["lowercase_and_strip"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace"] = text
    text = remove_unicode(text)
    history["remove_unicode"] = text
    text = remove_digits(text)
    history["remove_digits"] = text
    text = strip_punctuation(text)
    history["strip_punctuation"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace_after_punctuation"] = text
    tokens = tokenize(text)
    history["tokenize"] = tokens
    tokens = remove_stopwords(tokens)
    history["remove_stopwords"] = tokens
    history["stem_tokens"] = stem_tokens(tokens)
    return history


In [ ]:
# Store preprocessing variants on each filtered English comment
for comment in comments_flattened:
    original_text = comment.get("comment_text", "")
    topic_tokens = clean_for_topic_tokens(original_text)
    topic_stemmed_tokens = stem_tokens(topic_tokens)

    # Used for VADER and BERT (exact same preprocessign)
    comment["comment_text_sentiment_analysis"] = sentiment_analysis_clean(original_text)

    # Used for topic modelling
    comment["comment_text_topic"] = " ".join(topic_tokens)
    comment["comment_tokens_topic"] = topic_tokens

    # Used for topic modelling?
    comment["comment_text_topic_stemmed"] = " ".join(topic_stemmed_tokens)
    comment["comment_tokens_topic_stemmed"] = topic_stemmed_tokens
    comment["topic_token_count"] = len(topic_tokens)

print(f"Stored preprocessing variants for {len(comments_flattened)} English comments")


NameError: name 'clean_for_vader' is not defined

In [ ]:
# Demonstration of processing for random sample and report
RANDOM_TOTAL_EXAMPLES = 10
purify_random_samples = random.sample(english_comments, RANDOM_TOTAL_EXAMPLES)
for idx, text in enumerate(purify_random_samples):
    history = purify_text(text['comment_text'], True).items()
    print(f"\n[{idx+1}/{RANDOM_TOTAL_EXAMPLES}] {text['comment_text'][:100]}")
    for step, value in history:
        print(f"  [{step}] {value}")

In [ ]:
def counter_to_edge_rows(counter, source_field="source", target_field="target"):
    rows = []
    for pair, count in counter.items():
        source, target = pair
        rows.append({
            source_field: source,
            target_field: target,
            "count": count,
        })
    return rows

processed_video_data = {
    "comments": comments_flattened,
    "entity_counts": {
        "celebs": dict(celeb_counter),
        "brands": dict(brand_counter),
        "celeb_pairs": counter_to_edge_rows(celeb_pair_counter),
        "brand_celeb_pairs": counter_to_edge_rows(brand_celeb_counter),
    },
}

with open(PROCESSED_VIDEO_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(processed_video_data, f, ensure_ascii=False, indent=2)

print(f"Saved processed data to {PROCESSED_VIDEO_DATA_PATH}")


>